In [6]:
import boto3
import pandas as pd
import re
import io
import pyarrow
import pyarrow.parquet as pq
from datetime import datetime
import tempfile
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

In [5]:
BUCKET_NAME = 'darwin-raildata-mlops'
PROCESSED_PREFIX = 'darwin-kinesis-processed-rawdata'
LONG_DF_PREFIX = 'darwin-long-format-data'
MODEL_PREFIX = 'darwin-models'

s3 = boto3.client('s3')


def load_all_processed_csvs(bucket: str, prefix: str):
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    dfs = []
    for obj in response.get('Contents', []):
        key = obj['Key']
        if key.endswith('.csv'):
            csv_obj = s3.get_object(Bucket=bucket, Key=key)
            df = pd.read_csv(io.BytesIO(csv_obj['Body'].read()))
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

In [6]:
full_df = load_all_processed_csvs(BUCKET_NAME, PROCESSED_PREFIX)
if full_df.empty:
    print("No data after merge.")
        #return

In [7]:
full_df

,Pport_@xmlns,Pport_ns2,Pport_ns3,Pport_ns4,Pport_ns5,Pport_ns6,Pport_ns7,Pport_ns8,Pport_ns9,Pport_ns10,...,Pport_uR_schedule_OPIP_@act,Pport_uR_TS_Location_17_plat_@conf,Pport_uR_schedule_0_PP_13_@act,Pport_uR_schedule_0_PP_34_@can,Pport_uR_schedule_1_OR_@can,Pport_uR_schedule_1_DT_@can,Pport_uR_association_main_@wtd,Pport_uR_association_main_@ptd,Pport_uR_association_assoc_@wta,Pport_uR_association_assoc_@pta
0,http://www.thalesgroup.com/rtti/PushPort/v16,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Forec...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Stati...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/TDDat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,http://www.thalesgroup.com/rtti/PushPort/v16,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Forec...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Stati...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/TDDat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,http://www.thalesgroup.com/rtti/PushPort/v16,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Forec...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Stati...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/TDDat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,http://www.thalesgroup.com/rtti/PushPort/v16,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Forec...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Stati...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/TDDat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,http://www.thalesgroup.com/rtti/PushPort/v16,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Forec...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Stati...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/TDDat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2498,http://www.thalesgroup.com/rtti/PushPort/v16,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Forec...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Stati...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/TDDat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2499,http://www.thalesgroup.com/rtti/PushPort/v16,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Forec.

In [6]:
!pip install fsspec


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: C:\Users\personal\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [7]:
!pip install s3fs


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: C:\Users\personal\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [8]:
full_df.describe()

,Pport_@version,Pport_uR_TS_@rid,Pport_uR_TS_Location_0_length,Pport_uR_TS_Location_1_length,Pport_uR_TS_Location_2_length,Pport_uR_TS_Location_3_length,Pport_uR_TS_Location_4_length,Pport_uR_TS_Location_5_length,Pport_uR_TS_Location_6_length,Pport_uR_TS_Location_7_plat,...,Pport_uR_schedule_0_IP_4_@avgLoading,Pport_uR_schedule_0_IP_5_@avgLoading,Pport_uR_schedule_0_IP_6_@avgLoading,Pport_uR_schedule_0_IP_7_@avgLoading,Pport_uR_schedule_0_IP_8_@avgLoading,Pport_uR_schedule_0_IP_9_@avgLoading,Pport_uR_schedule_0_IP_10_@avgLoading,Pport_uR_schedule_0_IP_11_@avgLoading,Pport_uR_schedule_0_IP_12_@avgLoading,Pport_uR_schedule_0_DT_@avgLoading
count,2503.0,2.351000e+03,418.000000,418.000000,338.000000,293.000000,243.000000,208.000000,176.000000,88.000000,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
mean,16.0,2.025072e+14,5.334928,5.332536,5.396450,5.399317,5.358025,5.375000,5.443182,2.159091,...,14.0,13.0,14.0,14.0,14.0,22.0,18.0,18.0,16.0,16.0
std,0.0,8.464296e+05,2.960617,2.953995,2.917702,2.929701,2.965262,2.996576,3.064480,1.560182,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,16.0,2.025072e+14,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.000000,...,14.0,13.0,14.0,14.0,14.0,22.0,18.0,18.0,16.0,16.0
25%,16.0,2.025072e+14,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,1.000000,...,14.0,13.0,14.0,14.0,14.0,22.0,18.0,18.0,16.0,16.0
50%,16.0,2.025072e+14,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,2.000000,...,14.0,13.0,14.0,14.0,14.0,22.0,18.0,18.0,16.0,16.0
75%,16.0,2.025072e+14,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000,2.000000,...,14.0,13.0,14.0,14.0,14.0,22.0,18.0,18.0,16.0,16.0
max,16.0,2.025072e+14,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,9.000000,...,14.0,13.0,14.0,14.0,14.0,22.0,18.0,18.0,16.0,16.0


In [9]:
import pandas as pd
import re

def normalize_location_blocks(df: pd.DataFrame) -> pd.DataFrame:
    """
    Unflattens Darwin Rail data from wide format to long format.
    Each Location_n_ group becomes a separate row with common metadata.
    """
    df = df.copy()

    # Find all distinct location indices
    location_pattern = re.compile(r'Pport_uR_TS_Location_(\d+)_')
    location_indices = set()

    for col in df.columns:
        match = location_pattern.match(col)
        if match:
            location_indices.add(int(match.group(1)))

    # Get base columns (those not starting with Location_n_)
    base_cols = [col for col in df.columns if not location_pattern.match(col)]

    # Normalize each Location_n_ group
    long_rows = []
    for idx in sorted(location_indices):
        loc_prefix = f"Pport_uR_TS_Location_{idx}_"
        loc_cols = [col for col in df.columns if col.startswith(loc_prefix)]

        if not loc_cols:
            continue

        # Slice location data and rename
        temp = df[base_cols + loc_cols].copy()
        temp.columns = base_cols + [col.replace(loc_prefix, "") for col in loc_cols]
        temp["location_index"] = idx

        long_rows.append(temp)

    # Combine all normalized rows
    if not long_rows:
        return pd.DataFrame()  # Return empty if none found

    return pd.concat(long_rows, ignore_index=True)


In [10]:
long_df = normalize_location_blocks(full_df)

In [11]:
print(full_df.notna())

      Pport_@xmlns  Pport_ns2  Pport_ns3  Pport_ns4  Pport_ns5  Pport_ns6  \
0             True       True       True       True       True       True   
1             True       True       True       True       True       True   
2             True       True       True       True       True       True   
3             True       True       True       True       True       True   
4             True       True       True       True       True       True   
...            ...        ...        ...        ...        ...        ...   
2498          True       True       True       True       True       True   
2499          True       True       True       True       True       True   
2500          True       True       True       True       True       True   
2501          True       True       True       True       True       True   
2502          True       True       True       True       True       True   

      Pport_ns7  Pport_ns8  Pport_ns9  Pport_ns10  ...  \
0          True  

In [12]:
full_df['Pport_uR_TS_@uid'].head(10)

0    G57687
1    P18343
2    G57764
3    W09645
4    G31579
5    G55257
6    Y34125
7    G56132
8    G57685
9    Y72654
Name: Pport_uR_TS_@uid, dtype: object

In [13]:
full_df.columns.unique().tolist()

['Pport_@xmlns',
 'Pport_ns2',
 'Pport_ns3',
 'Pport_ns4',
 'Pport_ns5',
 'Pport_ns6',
 'Pport_ns7',
 'Pport_ns8',
 'Pport_ns9',
 'Pport_ns10',
 'Pport_ns11',
 'Pport_ns12',
 'Pport_@ts',
 'Pport_@version',
 'Pport_uR_@updateOrigin',
 'Pport_uR_TS_@rid',
 'Pport_uR_TS_@uid',
 'Pport_uR_TS_@ssd',
 'Pport_uR_TS_Location_0_@tpl',
 'Pport_uR_TS_Location_0_@wtp',
 'Pport_uR_TS_Location_0_pass_@et',
 'Pport_uR_TS_Location_0_pass_@delayed',
 'Pport_uR_TS_Location_0_pass_@src',
 'Pport_uR_TS_Location_0_length',
 'Pport_uR_TS_Location_1_@tpl',
 'Pport_uR_TS_Location_1_@wta',
 'Pport_uR_TS_Location_1_@wtd',
 'Pport_uR_TS_Location_1_@pta',
 'Pport_uR_TS_Location_1_@ptd',
 'Pport_uR_TS_Location_1_arr_@et',
 'Pport_uR_TS_Location_1_arr_@delayed',
 'Pport_uR_TS_Location_1_arr_@src',
 'Pport_uR_TS_Location_1_dep_@et',
 'Pport_uR_TS_Location_1_dep_@delayed',
 'Pport_uR_TS_Location_1_dep_@src',
 'Pport_uR_TS_Location_1_plat',
 'Pport_uR_TS_Location_1_length',
 'Pport_uR_TS_Location_2_@tpl',
 'Pport_uR_

In [14]:
long_df.columns.unique().tolist()

['Pport_@xmlns',
 'Pport_ns2',
 'Pport_ns3',
 'Pport_ns4',
 'Pport_ns5',
 'Pport_ns6',
 'Pport_ns7',
 'Pport_ns8',
 'Pport_ns9',
 'Pport_ns10',
 'Pport_ns11',
 'Pport_ns12',
 'Pport_@ts',
 'Pport_@version',
 'Pport_uR_@updateOrigin',
 'Pport_uR_TS_@rid',
 'Pport_uR_TS_@uid',
 'Pport_uR_TS_@ssd',
 'Pport_uR_TS_Location_@tpl',
 'Pport_uR_TS_Location_@wta',
 'Pport_uR_TS_Location_@wtd',
 'Pport_uR_TS_Location_@pta',
 'Pport_uR_TS_Location_@ptd',
 'Pport_uR_TS_Location_arr_@at',
 'Pport_uR_TS_Location_arr_@atClass',
 'Pport_uR_TS_Location_arr_@src',
 'Pport_uR_TS_Location_dep_@et',
 'Pport_uR_TS_Location_dep_@src',
 'Pport_uR_TS_Location_plat_@platsrc',
 'Pport_uR_TS_Location_plat_@conf',
 'Pport_uR_TS_Location_plat_#text',
 'Pport_uR_TS_Location_length',
 'Pport_uR_TS_Location_arr_@srcInst',
 'Pport_uR_TS_LateReason',
 'Pport_uR_TS_Location_arr_@et',
 'Pport_uR_TS_Location_plat',
 'Pport_uR_@requestSource',
 'Pport_uR_@requestID',
 'Pport_uR_TS_Location_@wtp',
 'Pport_uR_TS_Location_pass_

In [15]:
long_df['Pport_uR_TS_Location_@wtd'].head(10)

0         NaN
1       12:46
2         NaN
3    12:44:30
4       12:44
5         NaN
6         NaN
7         NaN
8         NaN
9         NaN
Name: Pport_uR_TS_Location_@wtd, dtype: object

In [17]:
long_df['Pport_uR_TS_Location_arr_@at'].tail(10)

227763      NaN
227764      NaN
227765      NaN
227766      NaN
227767      NaN
227768      NaN
227769      NaN
227770      NaN
227771    13:07
227772    13:07
Name: Pport_uR_TS_Location_arr_@at, dtype: object

In [16]:
full_df.shape

(2503, 2324)

In [17]:
len(long_df)

227773

In [41]:
long_df['Pport_uR_TS_Location_plat_'].head(10)

KeyError: 'Pport_uR_TS_Location_plat_'

In [37]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

def preprocess_for_arrival_delay(df):
    df = df.copy()

    # Drop rows where all columns are NaN or empty
    df = df.dropna(how='all')

    df = df.drop(
    columns=[col for col in df.columns if col == 'Pport_uR_TS_Location_plat' or col.startswith('Pport_uR_TS_Location_plat_')],
    errors='ignore'
    )


    #Drop rows where both values are missing
    #df = df.dropna(subset=['@wta', 'arr_@et'], how='all')  # drop if both are missing


    # Drop rows with missing key fields needed for delay calculation
    #df = df.dropna(subset=[
        #'Pport_uR_TS_@ssd',
        #'arr_@et',
        #'@wta'
    #])

    # Parse service date
    df['service_date'] = pd.to_datetime(df['Pport_uR_TS_@ssd'], errors='coerce').dt.date

    # Combine date + time to compute full datetime
    df['scheduled_arrival_dt'] = pd.to_datetime(
        df['service_date'].astype(str) + ' ' + df['Pport_uR_TS_Location_@wta'],
        errors='coerce'
    )

    df['actual_arrival_dt'] = pd.to_datetime(
        df['service_date'].astype(str) + ' ' + df['Pport_uR_TS_Location_arr_@at'],
        errors='coerce'
    )

    # Calculate arrival delay in minutes
    df['arrival_delay_minutes'] = (
        df['actual_arrival_dt'] - df['scheduled_arrival_dt']
    ).dt.total_seconds() / 60.0


    df_filtered = df[
        (df['arrival_delay_minutes'].notna()) &
        (df['arrival_delay_minutes'] >= -5) &
        (df['arrival_delay_minutes'] <= 120)
    ].copy()

    # Create categorical label for arrival status
    #df_filtered['arrival_status'] = df_filtered['arrival_delay_minutes'].apply(
        #lambda x: 'early' if x < 0 else ('delayed' if x > 0 else 'on_time')
    #)

    # Extract day of week
    df_filtered['day_of_week'] = pd.to_datetime(df_filtered['Pport_uR_TS_@ssd'], errors='coerce').dt.dayofweek

    return df_filtered

    # Encode station code
    #if '@tpl' not in df.columns:
        #raise ValueError("Missing '@tpl' column for station encoding")

    #encoder = LabelEncoder()
    #df['station_code'] = encoder.fit_transform(df['@tpl'].astype(str))

    #return df


In [30]:



def preprocess_for_arrival_delay(input_path, output_path):
    s3 = boto3.client("s3")
    
    # --- Parse input S3 path ---
    #bucket_in, key_in = input_path.replace("s3://", "").split("/", 1)
    
    # --- Read CSV from S3 ---
    #obj = s3.get_object(Bucket=bucket_in, Key=key_in)
    df = pd.read_parquet(input_path, storage_options={"anon": False})
    #df = df.dropna(how='all')  # Drop empty rows

    # Parse service date
    df['service_date'] = pd.to_datetime(df['Pport_uR_TS_@ssd'], errors='coerce').dt.date

    # Scheduled arrival datetime
    df['scheduled_arrival_dt'] = pd.to_datetime(
        df['service_date'].astype(str) + ' ' + df['Pport_uR_TS_Location_@wta'],
        errors='coerce'
    )

    # Actual arrival datetime
    df['actual_arrival_dt'] = pd.to_datetime(
        df['service_date'].astype(str) + ' ' + df['Pport_uR_TS_Location_arr_@at'],
        errors='coerce'
    )

    # Calculate delay in minutes
    df['arrival_delay_minutes'] = (
        df['actual_arrival_dt'] - df['scheduled_arrival_dt']
    ).dt.total_seconds() / 60.0

    # Filter by reasonable delays (-5 to 120 minutes)
    df_filtered = df[
       (df['arrival_delay_minutes'].notna()) &
        (df['arrival_delay_minutes'] >= -5) &
       (df['arrival_delay_minutes'] <= 120)
    ].copy()

    # Extract day of week
    df_filtered['day_of_week'] = pd.to_datetime(df['Pport_uR_TS_@ssd'], errors='coerce').dt.dayofweek

    # --- Save normalized Parquet back to S3 ---
    parquet_buffer = io.BytesIO()
    df_filtered.to_parquet(parquet_buffer, engine='pyarrow', index=False)
    
    bucket_out, key_out = output_path.replace("s3://", "").split("/", 1)
    s3.put_object(Bucket=bucket_out, Key=key_out, Body=parquet_buffer.getvalue())


In [31]:
preprocess_for_arrival_delay(
    input_path="s3://darwin-raildata-mlops/wide-to-long-transformation/",
    output_path="s3://darwin-raildata-mlops/transformed-data/preprocessed-features-data.parquet"
)

In [33]:
df = pd.read_parquet("s3://darwin-raildata-mlops/wide-to-long-transformation/", storage_options={"anon": False})

In [36]:
df['Pport_uR_TS_Location_arr_@at'].head(10)

0     None
1    12:45
2     None
3    12:43
4    12:45
5     None
6     None
7     None
8     None
9     None
Name: Pport_uR_TS_Location_arr_@at, dtype: object

In [40]:
transform_df = preprocess_for_arrival_delay(long_df)

In [42]:
transform_df['Pport_uR_TS_Location_plat_'].head(10)

KeyError: 'Pport_uR_TS_Location_plat_'

In [43]:
transform_df.tail(10)

,Pport_@xmlns,Pport_ns2,Pport_ns3,Pport_ns4,Pport_ns5,Pport_ns6,Pport_ns7,Pport_ns8,Pport_ns9,Pport_ns10,...,dep_@srcInst,dep_@etmin,arr_@etmin,dep_@wet,location_index,service_date,scheduled_arrival_dt,actual_arrival_dt,arrival_delay_minutes,day_of_week
227669,http://www.thalesgroup.com/rtti/PushPort/v16,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Forec...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Stati...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/TDDat...,...,NaN,NaN,NaN,NaN,90,2025-07-22,2025-07-22 13:07:30,2025-07-22 13:06:00,-1.5,1
227673,http://www.thalesgroup.com/rtti/PushPort/v16,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Forec...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Stati...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/TDDat...,...,NaN,NaN,NaN,NaN,90,2025-07-22,2025-07-22 13:07:30,2025-07-22 13:06:00,-1.5,1
227693,http://www.thalesgroup.com/rtti/PushPort/v16,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Forec...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Stati...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/TDDat...,...,NaN,NaN,NaN,NaN,90,2025-07-22,2025-07-22 13:04:30,2025-07-22 13:04:00,-0.5,1
227697,http://www.thalesgroup.com/rtti/PushPort/v16,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Forec...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Stati...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/TDDat...,...,NaN,NaN,NaN,NaN,90,2025-07-22,2025-07-22 13:05:30,2025-07-22 13:05:00,-0.5,1
227698,http://www.thalesgroup.com/rtti/PushPort/v16,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Forec...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Stati...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/TDDat...,...,NaN,NaN,NaN,NaN,90,2025-07-22,2025-07-22 13:03:30,2025-07-22 13:02:00,-1.5,1
227701,http://www.thalesgroup.com/rtti/PushPort/v16,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Forec...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Stati...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/Train...,http://www.thalesgroup.com/rtti/PushPort/TDDat...,...,NaN,NaN,NaN,NaN,90,2025-07-22,2025-07-22 13:06:30,2025-07-22 13:06:00,-0.5,1
227730,http://www.thalesgroup.com/rtti/PushPort/v16,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Sched...,http://www.thalesgroup.com/rtti/PushPort/Forma...,http://www.thalesgroup.com/rtti/PushPort/Forec

In [21]:
len(transform_df)

18291

In [22]:
transform_df.describe()

,Pport_@version,Pport_uR_TS_@rid,Pport_uR_TS_Location_length,Pport_uR_TS_LateReason,Pport_uR_schedule_0_@rid,Pport_uR_schedule_1_@rid,Pport_uR_association_main_@rid,Pport_uR_association_assoc_@rid,Pport_uR_formationLoading_@rid,Pport_uR_formationLoading_loading_0_#text,...,Pport_uR_schedule_0_IP_10_@avgLoading,Pport_uR_schedule_0_IP_11_@avgLoading,Pport_uR_schedule_0_IP_12_@avgLoading,Pport_uR_schedule_0_DT_@avgLoading,length,location_index,scheduled_arrival_dt,actual_arrival_dt,arrival_delay_minutes,day_of_week
count,18291.0,1.829100e+04,8190.000000,1456.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,18291.000000,18291,18291,18291.000000,18291.0
mean,16.0,2.025072e+14,6.155556,789.437500,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,45.000000,2025-07-22 12:59:49.701492480,2025-07-22 13:00:03.283582208,0.226368,1.0
min,16.0,2.025072e+14,2.000000,654.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.000000,2025-07-22 12:32:30,2025-07-22 12:43:00,-3.500000,1.0
25%,16.0,2.025072e+14,4.000000,705.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,22.000000,2025-07-22 12:58:30,2025-07-22 12:59:00,-0.500000,1.0
50%,16.0,2.025072e+14,5.000000,786.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,45.000000,2025-07-22 13:00:30,2025-07-22 13:00:00,-0.500000,1.0
75%,16.0,2.025072e+14,8.000000,904.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,68.000000,2025-07-22 13:02:30,2025-07-22 13:02:00,0.500000,1.0
max,16.0,2.025072e+14,12.000000,913.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,90.000000,2025-07-22 13:08:30,2025-07-22 13:07:00,23.500000,1.0
std,0.0,7.347806e+05,3.299734,95.654484,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,26.268569,NaN,NaN,2.849491,0.0


In [23]:
transform_df.columns.tolist()

['Pport_@xmlns',
 'Pport_ns2',
 'Pport_ns3',
 'Pport_ns4',
 'Pport_ns5',
 'Pport_ns6',
 'Pport_ns7',
 'Pport_ns8',
 'Pport_ns9',
 'Pport_ns10',
 'Pport_ns11',
 'Pport_ns12',
 'Pport_@ts',
 'Pport_@version',
 'Pport_uR_@updateOrigin',
 'Pport_uR_TS_@rid',
 'Pport_uR_TS_@uid',
 'Pport_uR_TS_@ssd',
 'Pport_uR_TS_Location_@tpl',
 'Pport_uR_TS_Location_@wta',
 'Pport_uR_TS_Location_@wtd',
 'Pport_uR_TS_Location_@pta',
 'Pport_uR_TS_Location_@ptd',
 'Pport_uR_TS_Location_arr_@at',
 'Pport_uR_TS_Location_arr_@atClass',
 'Pport_uR_TS_Location_arr_@src',
 'Pport_uR_TS_Location_dep_@et',
 'Pport_uR_TS_Location_dep_@src',
 'Pport_uR_TS_Location_plat_@platsrc',
 'Pport_uR_TS_Location_plat_@conf',
 'Pport_uR_TS_Location_plat_#text',
 'Pport_uR_TS_Location_length',
 'Pport_uR_TS_Location_arr_@srcInst',
 'Pport_uR_TS_LateReason',
 'Pport_uR_TS_Location_arr_@et',
 'Pport_uR_TS_Location_plat',
 'Pport_uR_@requestSource',
 'Pport_uR_@requestID',
 'Pport_uR_TS_Location_@wtp',
 'Pport_uR_TS_Location_pass_

In [27]:
import io
import joblib
import boto3
import pandas as pd
from sklearn.feature_extraction import DictVectorizer


def prepare_features(input_path, output_path, train_ratio=0.8):
    """
    Prepares features for training and validation with a flexible train/val split.
    Saves output to S3 as joblib files.
    """

    s3 = boto3.client("s3")

    # Read parquet from S3
    df = pd.read_parquet(input_path, storage_options={"anon": False})

    # Ensure required columns exist
    if 'day_of_week' not in df.columns or 'Pport_uR_TS_Location_@tpl' not in df.columns:
        raise ValueError("Missing required columns in DataFrame")

    # Train/val split
    split_index = int(len(df) * train_ratio)
    df_train = df.iloc[:split_index].copy()
    df_val = df.iloc[split_index:].copy()

    # Combined categorical feature
    df_train['day_of_week_Pport_uR_TS_Location_@tpl'] = (
        df_train['day_of_week'].astype(str) + '_' + df_train['Pport_uR_TS_Location_@tpl'].astype(str)
    )
    df_val['day_of_week_Pport_uR_TS_Location_@tpl'] = (
        df_val['day_of_week'].astype(str) + '_' + df_val['Pport_uR_TS_Location_@tpl'].astype(str)
    )

    # Vectorize
    categorical = ['day_of_week_Pport_uR_TS_Location_@tpl']
    target = 'arrival_delay_minutes'

    dv = DictVectorizer()
    X_train = dv.fit_transform(df_train[categorical].to_dict(orient='records'))
    X_val = dv.transform(df_val[categorical].to_dict(orient='records'))

    y_train = df_train[target].values
    y_val = df_val[target].values

    # Save to S3
    bucket_out, key_prefix = output_path.replace("s3://", "").split("/", 1)

    for obj, name in [
        (X_train, "X_train.joblib"),
        (y_train, "y_train.joblib"),
        (X_val, "X_val.joblib"),
        (y_val, "y_val.joblib"),
        (dv, "dict_vectorizer.joblib")
    ]:
        buffer = io.BytesIO()
        joblib.dump(obj, buffer)
        buffer.seek(0)
        s3.upload_fileobj(buffer, bucket_out, f"{key_prefix}/{name}")

    print(f"✅ Features saved to {output_path}")

    return X_train, y_train, X_val, y_val, dv


In [ ]:
df = pd.read_parquet(input_path, storage_options={"anon": False})

In [28]:
X_train, y_train, X_val, y_val, dv = prepare_features(input_path = "s3://darwin-raildata-mlops/transformed-data/preprocessed-features-data/",train_ratio=0.8,output_path= "s3://darwin-raildata-mlops/feature-eng-data/")


ValueError: Missing required columns in DataFrame

In [32]:
import pandas as pd

df = pd.read_parquet("s3://darwin-raildata-mlops/transformed-data/preprocessed-features-data/")
print(df.columns.tolist())
print(df.head())


[]
Empty DataFrame
Columns: []
Index: []


In [28]:
!pip install xgboost



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: C:\Users\personal\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [29]:
!pip install hyperopt


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: C:\Users\personal\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [30]:
!pip install -U scikit-learn


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: C:\Users\personal\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [41]:
!pip install mlflow


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: C:\Users\personal\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope
import numpy as np
import warnings

warnings.filterwarnings("ignore")

def find_best_model_params(X_train, y_train, X_val, y_val, max_evals=30, verbose=True):
    model_classes = {
        "RandomForest": RandomForestRegressor,
        "XGBoost": XGBRegressor,
        "LinearRegression": LinearRegression
    }

    search_spaces = {
        "RandomForest": {
            "n_estimators": scope.int(hp.quniform("n_estimators", 50, 200, 10)),
            "max_depth": scope.int(hp.quniform("max_depth", 4, 30, 1)),
        },
        "XGBoost": {
            "n_estimators": scope.int(hp.quniform("n_estimators", 50, 200, 10)),
            "max_depth": scope.int(hp.quniform("max_depth", 3, 10, 1)),
            "learning_rate": hp.loguniform("learning_rate", -3, 0),
        },
        "LinearRegression": {}  # No hyperparameters to tune
    }

    best_rmse = float("inf")
    best_model = None
    best_params = {}

    for model_name, model_class in model_classes.items():
        if verbose:
            print(f"\n🔍 Tuning {model_name}...")

        if model_name == "LinearRegression":
            model = model_class()
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            mse = mean_squared_error(y_val, y_pred)
            rmse = np.sqrt(mse)


            if verbose:
                print(f"📊 RMSE for LinearRegression: {rmse:.4f}")

            if rmse < best_rmse:
                best_rmse = rmse
                best_model = model_name
                best_params = {}
            continue

        def objective(params):
            model = model_class(**params)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            mse = mean_squared_error(y_val, y_pred)
            rmse = np.sqrt(mse)
            return {"loss": rmse, "status": STATUS_OK}

        trials = Trials()
        best = fmin(
            fn=objective,
            space=search_spaces[model_name],
            algo=tpe.suggest,
            max_evals=max_evals,
            trials=trials,
            rstate=np.random.default_rng(42)
        )

        # Cast int-type hyperparams from float to int
        best_casted = {k: int(v) if k in ["n_estimators", "max_depth"] else v for k, v in best.items()}

        model = model_class(**best_casted)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        mse = mean_squared_error(y_val, y_pred)
        rmse = np.sqrt(mse)

        if verbose:
            print(f"📊 RMSE for {model_name}: {rmse:.4f} | Params: {best_casted}")

        if rmse < best_rmse:
            best_rmse = rmse
            best_model = model_name
            best_params = best_casted

    if verbose:
        print(f"\n✅ Best Model: {best_model}")
        print(f"✅ Best Params: {best_params}")
        print(f"✅ Best RMSE: {best_rmse:.4f}")

    return best_model, best_params, best_rmse


In [27]:
best_model, best_params, best_rmse = find_best_model_params(X_train, y_train, X_val, y_val)


🔍 Tuning RandomForest...
100%|███████████████████████████████████| 30/30 [01:08<00:00,  2.29s/trial, best loss: 0.7649841367514362]
📊 RMSE for RandomForest: 0.7649 | Params: {'max_depth': 30, 'n_estimators': 190}

🔍 Tuning XGBoost...
100%|██████████████████████████████████| 30/30 [00:06<00:00,  4.98trial/s, best loss: 0.23499605915742583]
📊 RMSE for XGBoost: 0.2350 | Params: {'learning_rate': np.float64(0.658883492655059), 'max_depth': 7, 'n_estimators': 100}

🔍 Tuning LinearRegression...
📊 RMSE for LinearRegression: 0.2350

✅ Best Model: XGBoost
✅ Best Params: {'learning_rate': np.float64(0.658883492655059), 'max_depth': 7, 'n_estimators': 100}
✅ Best RMSE: 0.2350


In [28]:
import mlflow
import boto3
import pickle
import os
import numpy as np
import mlflow.sklearn
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Xgboost-hyperopt")
def train_model(X_train, y_train, X_val, y_val, model_class, best_params, bucket_name, model_key, dv):
    """
    Trains a model using best parameters, logs with MLflow, and uploads to S3.

    Args:
        X_train, y_train: Training data
        X_val, y_val: Validation data
        model_class (str or class): Model class name or class itself
        best_params (dict): Hyperparameters
        bucket_name (str): S3 bucket name
        model_key (str): Path to save model in S3
        dv (DictVectorizer): Fitted vectorizer to save with model

    Returns:
        model: Trained model
        rmse: Root Mean Squared Error on validation set
    """

    # Convert model name string to class if needed
    if isinstance(model_class, str):
        model_classes = {
            "RandomForest": RandomForestRegressor,
            "LinearRegression": LinearRegression,
            "XGBoost": XGBRegressor
        }
        model_class = model_classes[model_class]

    os.makedirs("/tmp", exist_ok=True)

    # Enable MLflow autologging for sklearn
    mlflow.sklearn.autolog()
    with mlflow.start_run():
        mlflow.set_tag("developer", "Rita")
        mlflow.log_param("model_class", model_class.__name__)
        mlflow.log_params(best_params)
        
        # Train
        model = model_class(**best_params)
        model.fit(X_train, y_train)

        # Evaluate
        y_pred = model.predict(X_val)
        mse = mean_squared_error(y_val, y_pred)
        rmse = np.sqrt(mse)

        # Log the metric
        mlflow.log_metric("rmse", rmse)

        run_id = mlflow.active_run().info.run_id
        
        # Save model and preprocessor
        local_path = "/tmp/model.pkl"
        with open(local_path, "wb") as f_out:
            pickle.dump((dv, model), f_out)

        
        mlflow.sklearn.log_model(model, artifact_path="models_pickle")


        # Upload to S3
        s3 = boto3.client("s3")
        s3.upload_file(local_path, bucket_name, model_key)

        print(f"✅ Model uploaded to s3://{bucket_name}/{model_key}")
        print(f"📈 Validation RMSE: {rmse:.4f}")
        print(f"📈 run_id: {run_id}")

    return model, rmse, run_id


In [29]:

model = train_model(
    X_train, y_train, X_val, y_val,
    model_class=best_model,
    best_params=best_params,
    bucket_name='darwin-raildata-mlops',
    model_key='models/best_model.pkl',
    dv=dv
)


2025/08/02 10:43:43 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.3.0 <= scikit-learn <= 1.7.0, but the installed version is 1.7.1. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2025/08/02 10:43:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/08/02 10:44:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ Model uploaded to s3://darwin-raildata-mlops/models/best_model.pkl
📈 Validation RMSE: 0.2350
📈 run_id: c12caa12c4d84ccb891830f10214001c
🏃 View run languid-mole-623 at: http://127.0.0.1:5000/#/experiments/2/runs/c12caa12c4d84ccb891830f10214001c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [30]:
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType
from datetime import datetime

def manage_and_register_best_model(experiment_name, metric_threshold, top_k, model_name, stage="Production"):
    """
    Finds best run under an RMSE threshold, registers it, and transitions it to the given stage.

    If no runs meet the RMSE threshold, fallback to top K best runs.

    Args:
        experiment_name (str): MLflow experiment name
        metric_threshold (float): RMSE threshold
        top_k (int): Max runs to consider if threshold not met
        model_name (str): Name for registered model
        stage (str): Target stage (e.g. 'Production', 'Staging')

    Returns:
        run_id (str): ID of best run
    """
    client = MlflowClient()
    mlflow.set_experiment(experiment_name)
    experiment = client.get_experiment_by_name(experiment_name)

    if experiment is None:
        raise ValueError(f"Experiment '{experiment_name}' not found.")
    experiment_id = experiment.experiment_id

    # First try to get runs below the threshold
    runs = client.search_runs(
        experiment_ids=[experiment_id],
        filter_string=f"metrics.rmse < {metric_threshold}",
        run_view_type=ViewType.ACTIVE_ONLY,
        max_results=top_k,
        order_by=["metrics.rmse ASC"]
    )

    # Fallback to top_k lowest RMSE runs if none meet the threshold
    if not runs:
        print("⚠️ No runs below threshold found. Falling back to top K lowest RMSE runs.")
        runs = client.search_runs(
            experiment_ids=[experiment_id],
            run_view_type=ViewType.ACTIVE_ONLY,
            max_results=top_k,
            order_by=["metrics.rmse ASC"]
        )

    if not runs:
        raise ValueError("❌ No runs found in the experiment at all.")

    # Get best run
    best_run = runs[0]
    run_id = best_run.info.run_id
    rmse = best_run.data.metrics.get("rmse")

    print(f"✅ Best run: {run_id} with RMSE: {rmse:.4f}")

    # Register the model
    model_uri = f"runs:/{run_id}/models_pickle"
    result = mlflow.register_model(model_uri=model_uri, name=model_name)
    model_version = result.version

    # Transition stage
    client.transition_model_version_stage(
        name=model_name,
        version=model_version,
        stage=stage,
        archive_existing_versions=True
    )

    # Update description
    today = datetime.today().date()
    client.update_model_version(
        name=model_name,
        version=model_version,
        description=f"Model version {model_version} transitioned to {stage} on {today}"
    )

    print(f"🚀 Model registered and promoted to '{stage}' stage: version {model_version}")

    return run_id



In [32]:
run_id = manage_and_register_best_model(
    experiment_name="Xgboost-hyperopt",
    metric_threshold=7.0,
    top_k=5,
    model_name="arrival-delay-predictor",
    stage="Production"
)


Successfully registered model 'arrival-delay-predictor'.
2025/08/02 11:11:07 WARNING mlflow.tracking._model_registry.fluent: Run with id c12caa12c4d84ccb891830f10214001c has no artifacts at artifact path 'models_pickle', registering model based on models:/m-39d7b0c994dc41aebbd82960d7fdf8f2 instead
2025/08/02 11:11:07 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: arrival-delay-predictor, version 1


✅ Best run: c12caa12c4d84ccb891830f10214001c with RMSE: 0.2350


Created version '1' of model 'arrival-delay-predictor'.


🚀 Model registered and promoted to 'Production' stage: version 1
